## MODELING - DECISION TREE VS KNN(END TO END)

In [1]:
import pandas as pd
import numpy as np
import re
from langdetect import detect
from textblob import TextBlob
import nltk
import unicodedata
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from rake_nltk import Rake
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora, models
import spacy
from sqlalchemy import create_engine, text
import os

In [2]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from scipy.sparse import hstack


In [13]:
df = pd.read_csv("scored_reviews.csv")

In [14]:
df = df[df["sentiment_combined"].notna()]
y = df["sentiment_combined"]


In [19]:
X_text = df["content"].astype(str)
X_num = df[["score","thumbs_up"]].fillna(0)


In [20]:
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=3) 

In [21]:
X_text_tfidf = tfidf.fit_transform(X_text)
from scipy import sparse
X_full = hstack([X_text_tfidf, sparse.csr_matrix(X_num.values)])


In [26]:
df['binary_sentiment'] = df['sentiment_vader_label'].map({
    'positive': 1,
    'neutral': 0
})

In [29]:
y = df["sentiment_combined"].apply(lambda x: 1 if x=="positive" else 0)

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42, stratify=y)

In [31]:
dt = DecisionTreeClassifier(random_state=42)
dt_params = {
    "criterion": ["gini","entropy"],
    "max_depth": [None, 5,10,20],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4]
}

In [32]:
dt_grid = GridSearchCV(dt, dt_params, cv=StratifiedKFold(5), scoring='f1_macro', n_jobs=-1, verbose=1)
dt_grid.fit(X_train, y_train)
print("DT best:", dt_grid.best_params_)
dt_pred = dt_grid.predict(X_test)
print("DT Test f1:", f1_score(y_test, dt_pred, average="macro"))
print(classification_report(y_test, dt_pred))
print(confusion_matrix(y_test, dt_pred))


Fitting 5 folds for each of 72 candidates, totalling 360 fits
DT best: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
DT Test f1: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        64

    accuracy                           1.00        64
   macro avg       1.00      1.00      1.00        64
weighted avg       1.00      1.00      1.00        64

[[64]]


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [33]:
knn = KNeighborsClassifier()
knn_params = {
    "n_neighbors": [3,5,7],
    "weights": ["uniform","distance"],
    "p": [1,2]
}

In [34]:
knn_grid = GridSearchCV(knn, knn_params, cv=StratifiedKFold(5), scoring='f1_macro', n_jobs=-1, verbose=1)
knn_grid.fit(X_train, y_train)
print("KNN best:", knn_grid.best_params_)
knn_pred = knn_grid.predict(X_test)
print("KNN Test f1:", f1_score(y_test, knn_pred, average="macro"))
print(classification_report(y_test, knn_pred))
print(confusion_matrix(y_test, knn_pred))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
KNN best: {'n_neighbors': 3, 'p': 1, 'weights': 'uniform'}
KNN Test f1: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        64

    accuracy                           1.00        64
   macro avg       1.00      1.00      1.00        64
weighted avg       1.00      1.00      1.00        64

[[64]]


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
